In [44]:
# import libraries
import os
import json
import time
import boto3
import sagemaker

from sagemaker.workflow.pipeline_context import PipelineSession
from sagemaker.workflow.pipeline import Pipeline
from sagemaker.workflow.parameters import ParameterString, ParameterFloat, ParameterInteger
from sagemaker.workflow.steps import ProcessingStep, TrainingStep
from sagemaker.workflow.step_collections import RegisterModel
from sagemaker.workflow.properties import PropertyFile
from sagemaker.workflow.condition_step import ConditionStep
from sagemaker.workflow.conditions import ConditionGreaterThanOrEqualTo
from sagemaker.workflow.fail_step import FailStep
from sagemaker.workflow.functions import JsonGet, Join
from sagemaker.workflow.execution_variables import ExecutionVariables

from sagemaker.processing import ProcessingInput, ProcessingOutput, Processor
from sagemaker.sklearn.processing import SKLearnProcessor
from sagemaker.inputs import TrainingInput
from sagemaker.estimator import Estimator
from sagemaker.image_uris import retrieve
from sagemaker.model_metrics import ModelMetrics, MetricsSource
from sagemaker.model import ModelPackage

# AWS setup
region = boto3.Session().region_name
sess = sagemaker.Session()
pipeline_sess = PipelineSession()
role = sagemaker.get_execution_role()
bucket = sess.default_bucket()
sm = boto3.client("sagemaker", region_name=region)

print("Region:", region)
print("Bucket:", bucket)

# set endpoint
PROD_ENDPOINT_NAME = "crema-d-emotion-endpoint-2026-02-20-00-57"

# model group
MODEL_PACKAGE_GROUP = "crema-d-emotion-recognition-models-2"

# Local artifact input file
LOCAL_PROCESSED_CSV = "artifacts/emovo_features.csv"

Region: us-east-1
Bucket: sagemaker-us-east-1-472875368112


In [45]:
%%writefile preprocess_emovo_unit_tests.py
import os
import json
import argparse
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder

FEATURES = [
    "rms_mean",
    "pitch_mean",
    "pitch_std",
    "spectral_centroid_mean",
    "mfcc_1_mean",
    "mfcc_2_mean",
    "mfcc_3_mean",
    "tempo",
]

def fail(msg: str):
    raise RuntimeError(msg)

def main():
    parser = argparse.ArgumentParser()
    parser.add_argument("--input-dir", type=str, default="/opt/ml/processing/input")
    parser.add_argument("--output-dir", type=str, default="/opt/ml/processing/output")
    parser.add_argument("--missing-threshold", type=float, default=0.05)
    parser.add_argument("--seed", type=int, default=42)
    args = parser.parse_args()

    in_csv = os.path.join(args.input_dir, "emovo_features.csv")
    if not os.path.exists(in_csv):
        cands = [f for f in os.listdir(args.input_dir) if f.endswith(".csv")]
        if not cands:
            fail(f"No CSV found in {args.input_dir}")
        in_csv = os.path.join(args.input_dir, cands[0])

    df = pd.read_csv(in_csv)

    # schema unit test
    if "emotion" not in df.columns:
        fail("Unit test failed: missing required column 'emotion'.")

    missing_feats = [c for c in FEATURES if c not in df.columns]
    if missing_feats:
        fail(f"Unit test failed: missing required feature columns: {missing_feats}")

    # mapping split column value of Hold Out to test
    if "split" in df.columns:
        df["split"] = (
            df["split"]
            .astype(str)
            .str.strip()
            .str.lower()
            .replace({"holdout_test": "test"})
        )
        
    # tempo
    df["tempo"] = pd.to_numeric(df["tempo"], errors="coerce")

    # types unit test
    for c in FEATURES:
        df[c] = pd.to_numeric(df[c], errors="coerce")

    # missingness checks
    miss = df[FEATURES].isna().mean().to_dict()
    offenders = {k: v for k, v in miss.items() if v > args.missing_threshold}
    if offenders:
        fail(f"Unit test failed: missingness above threshold {args.missing_threshold:.2%}: {offenders}")

    # remove NaNs
    df = df.replace([np.inf, -np.inf], np.nan).dropna(subset=FEATURES + ["emotion"]).copy()

    if len(df) < 200:
        fail(f"Unit test failed: too few rows after cleaning ({len(df)}).")

    # label encoding
    le = LabelEncoder()
    df["emotion_encoded"] = le.fit_transform(df["emotion"].astype(str))

    # classes unit test
    n_classes = df["emotion_encoded"].nunique()
    if n_classes < 2:
        fail(f"Unit test failed: need >=2 classes, found {n_classes}.")

    os.makedirs(args.output_dir, exist_ok=True)

    # save mapping
    mapping = {cls: int(idx) for cls, idx in zip(le.classes_, le.transform(le.classes_))}
    with open(os.path.join(args.output_dir, "label_mapping.json"), "w") as f:
        json.dump(mapping, f, indent=2)

    # Use existing split if present, otherwise create split
    use_existing_split = "split" in df.columns and df["split"].isin(["train", "val", "test"]).all()

    if use_existing_split:
        train_df = df[df["split"] == "train"].copy()
        val_df   = df[df["split"] == "val"].copy()
        test_df  = df[df["split"] == "test"].copy()
        if min(len(train_df), len(val_df), len(test_df)) == 0:
            use_existing_split = False
    # use 80/10/10
    if not use_existing_split:
        tmp_train, test_df = train_test_split(
            df, test_size=0.10, random_state=args.seed, stratify=df["emotion_encoded"]
        )
        train_df, val_df = train_test_split(
            tmp_train, test_size=0.1111, random_state=args.seed, stratify=tmp_train["emotion_encoded"]
        )

    # XGBoost format
    def to_xgb(dfin: pd.DataFrame) -> pd.DataFrame:
        out = dfin[["emotion_encoded"] + FEATURES].copy()
        out = out.replace([np.inf, -np.inf], np.nan).dropna()
        return out

    train_xgb = to_xgb(train_df)
    val_xgb   = to_xgb(val_df)
    test_xgb  = to_xgb(test_df)

    # write outputs
    train_dir = os.path.join(args.output_dir, "train")
    val_dir   = os.path.join(args.output_dir, "validation")
    test_dir  = os.path.join(args.output_dir, "test")
    for d in [train_dir, val_dir, test_dir]:
        os.makedirs(d, exist_ok=True)

    train_xgb.to_csv(os.path.join(train_dir, "train.csv"), index=False, header=False)
    val_xgb.to_csv(os.path.join(val_dir, "validation.csv"), index=False, header=False)
    test_xgb.to_csv(os.path.join(test_dir, "test.csv"), index=False, header=False)

    # small report for debugging
    report = {
        "rows_in": int(len(df)),
        "rows_train": int(len(train_xgb)),
        "rows_val": int(len(val_xgb)),
        "rows_test": int(len(test_xgb)),
        "num_classes": int(n_classes),
        "missingness": miss,
        "labels": mapping,
    }
    with open(os.path.join(args.output_dir, "preprocess_report.json"), "w") as f:
        json.dump(report, f, indent=2)

    print("Preprocess + unit tests passed.")
    print(json.dumps(report, indent=2))

if __name__ == "__main__":
    main()

Overwriting preprocess_emovo_unit_tests.py


In [46]:
%%writefile evaluate_xgb_emovo.py
import sys, subprocess
subprocess.check_call([sys.executable, "-m", "pip", "install", "xgboost==1.7.6"])

import os
import json
import tarfile
import argparse
import numpy as np
import pandas as pd
import xgboost as xgb
from sklearn.metrics import accuracy_score, classification_report

def find_model_file(extract_dir: str) -> str:
    preferred = {"xgboost-model", "model.bin", "model"}
    found = []
    for root, _, files in os.walk(extract_dir):
        for f in files:
            full = os.path.join(root, f)
            found.append(full)
            if f in preferred:
                return full

    nonempty = [p for p in found if os.path.getsize(p) > 0]
    if len(nonempty) == 1:
        return nonempty[0]

    raise FileNotFoundError(f"Could not locate model file. Sample: {nonempty[:25]}")

def resolve_tarball(path: str) -> str:
    if os.path.isfile(path) and path.endswith((".tar.gz", ".tgz")):
        return path

    if os.path.isdir(path):
        candidate = os.path.join(path, "model.tar.gz")
        if os.path.exists(candidate):
            return candidate

        tgzs = []
        for root, _, files in os.walk(path):
            for f in files:
                if f.endswith((".tar.gz", ".tgz")):
                    tgzs.append(os.path.join(root, f))
        if not tgzs:
            raise FileNotFoundError(f"No .tar.gz/.tgz found under directory: {path}")

        tgzs.sort(key=lambda p: os.path.getsize(p), reverse=True)
        return tgzs[0]

    raise FileNotFoundError(f"Model artifact path not found: {path}")

def main():
    parser = argparse.ArgumentParser()
    parser.add_argument("--model-artifact", type=str, required=True)
    parser.add_argument("--test-path", type=str, required=True)
    parser.add_argument("--output-dir", type=str, default="/opt/ml/processing/evaluation")
    args = parser.parse_args()

    os.makedirs(args.output_dir, exist_ok=True)

    tar_path = resolve_tarball(args.model_artifact)
    print("tarball:", tar_path)

    extract_dir = "/tmp/model_artifact"
    os.makedirs(extract_dir, exist_ok=True)

    with tarfile.open(tar_path, "r:gz") as tar:
        tar.extractall(path=extract_dir)

    model_file = find_model_file(extract_dir)
    print("model file:", model_file)

    booster = xgb.Booster()
    booster.load_model(model_file)

    test = pd.read_csv(args.test_path, header=None)
    y_true = test.iloc[:, 0].astype(int).values
    X = test.iloc[:, 1:].values

    dtest = xgb.DMatrix(X)
    y_pred = booster.predict(dtest)
    y_pred = np.array(y_pred).astype(int)

    acc = float(accuracy_score(y_true, y_pred))
    report = classification_report(
        y_true,
        y_pred,
        output_dict=True
    )

    evaluation = {
        "classification_metrics": {
            "accuracy": {"value": acc}
        },
        "classification_report": report
    }

    out_path = os.path.join(args.output_dir, "evaluation.json")
    with open(out_path, "w") as f:
        json.dump(evaluation, f, indent=2)

    print("Saved:", out_path)
    print(json.dumps(evaluation, indent=2))

if __name__ == "__main__":
    main()

Overwriting evaluate_xgb_emovo.py


In [47]:
PREFIX_DEFAULT = "ser-cicd-emovo"
raw_key = f"{PREFIX_DEFAULT}/raw/emovo_features.csv"

boto3.client("s3", region_name=region).upload_file(LOCAL_PROCESSED_CSV, bucket, raw_key)
RAW_EMOVO_S3 = f"s3://{bucket}/{raw_key}"

print("Uploaded EMOVO features CSV to:", RAW_EMOVO_S3)

Uploaded EMOVO features CSV to: s3://sagemaker-us-east-1-472875368112/ser-cicd-emovo/raw/emovo_features.csv


In [48]:
# Pipeline parameters
p_prefix = ParameterString("Prefix", default_value=PREFIX_DEFAULT)
p_processing_instance = ParameterString("ProcessingInstanceType", default_value="ml.t3.large")
p_training_instance   = ParameterString("TrainingInstanceType", default_value="ml.m5.xlarge")
p_accuracy_threshold  = ParameterFloat("AccuracyThreshold", default_value=0.70)

p_num_round = ParameterInteger("NumRound", default_value=100)
p_max_depth = ParameterInteger("MaxDepth", default_value=6)
p_eta       = ParameterFloat("Eta", default_value=0.3)

p_model_package_group = ParameterString("ModelPackageGroupName", default_value=MODEL_PACKAGE_GROUP)

# images/processors
xgb_image = retrieve(framework="xgboost", region=region, version="1.7-1")

sk_processor = SKLearnProcessor(
    framework_version="1.2-1",
    role=role,
    instance_type=p_processing_instance,
    instance_count=1,
    sagemaker_session=pipeline_sess,
)

# preprocess and unit test
processing_base = Join(on="", values=[
    "s3://", bucket, "/", p_prefix, "/processing/", ExecutionVariables.PIPELINE_EXECUTION_ID
])

preprocess_step = ProcessingStep(
    name="PreprocessEmovoAndUnitTests",
    processor=sk_processor,
    code="preprocess_emovo_unit_tests.py",
    inputs=[
        ProcessingInput(
            source=Join(on="", values=["s3://", bucket, "/", p_prefix, "/raw/emovo_features.csv"]),
            destination="/opt/ml/processing/input",
            input_name="raw"
        )
    ],
    outputs=[
        ProcessingOutput(
            output_name="train",
            source="/opt/ml/processing/output/train",
            destination=Join(on="", values=[processing_base, "/train"]),
        ),
        ProcessingOutput(
            output_name="validation",
            source="/opt/ml/processing/output/validation",
            destination=Join(on="", values=[processing_base, "/validation"]),
        ),
        ProcessingOutput(
            output_name="test",
            source="/opt/ml/processing/output/test",
            destination=Join(on="", values=[processing_base, "/test"]),
        ),
        ProcessingOutput(
            output_name="meta",
            source="/opt/ml/processing/output",
            destination=Join(on="", values=[processing_base, "/meta"]),
        ),
    ],
    job_arguments=[
        "--input-dir", "/opt/ml/processing/input",
        "--output-dir", "/opt/ml/processing/output",
        "--missing-threshold", "0.05",
        "--seed", "42",
    ],
)

train_s3 = preprocess_step.properties.ProcessingOutputConfig.Outputs["train"].S3Output.S3Uri
val_s3   = preprocess_step.properties.ProcessingOutputConfig.Outputs["validation"].S3Output.S3Uri
test_s3  = preprocess_step.properties.ProcessingOutputConfig.Outputs["test"].S3Output.S3Uri

# train model
xgb_estimator = Estimator(
    image_uri=xgb_image,
    role=role,
    instance_count=1,
    instance_type=p_training_instance,
    volume_size=50,
    max_run=3600,
    output_path=Join(on="", values=["s3://", bucket, "/", p_prefix, "/training-output"]),
    sagemaker_session=pipeline_sess,
    base_job_name="ser-xgb-train-emovo",
)

xgb_estimator.set_hyperparameters(
    objective="multi:softmax",
    num_class=6,
    max_depth=p_max_depth,
    eta=p_eta,
    subsample=0.8,
    colsample_bytree=0.8,
    eval_metric="mlogloss",
    num_round=p_num_round,
)

train_step = TrainingStep(
    name="TrainXGBoostOnEmovo",
    estimator=xgb_estimator,
    inputs={
        "train": TrainingInput(s3_data=train_s3, content_type="text/csv"),
        "validation": TrainingInput(s3_data=val_s3, content_type="text/csv"),
    }
)

# evaluate model
eval_processor = SKLearnProcessor(
    framework_version="1.2-1",
    role=role,
    instance_type=p_processing_instance,
    instance_count=1,
    sagemaker_session=pipeline_sess,
)

evaluation_report = PropertyFile(
    name="EvaluationReport",
    output_name="evaluation",
    path="evaluation.json"
)

evaluation_base = Join(on="", values=[
    "s3://", bucket, "/", p_prefix, "/evaluation/", ExecutionVariables.PIPELINE_EXECUTION_ID
])

eval_step = ProcessingStep(
    name="EvaluateCandidateOnEmovo",
    processor=eval_processor,
    code="evaluate_xgb_emovo.py",
    inputs=[
        ProcessingInput(
            source=train_step.properties.ModelArtifacts.S3ModelArtifacts,
            destination="/opt/ml/processing/model",
            input_name="model"
        ),
        ProcessingInput(
            source=test_s3,
            destination="/opt/ml/processing/test",
            input_name="test"
        ),
    ],
    outputs=[
        ProcessingOutput(
            output_name="evaluation",
            source="/opt/ml/processing/evaluation",
            destination=evaluation_base
        )
    ],
    job_arguments=[
        "--model-artifact", "/opt/ml/processing/model",
        "--test-path", "/opt/ml/processing/test/test.csv",
        "--output-dir", "/opt/ml/processing/evaluation"
    ],
    property_files=[evaluation_report],
)

accuracy_val = JsonGet(
    step_name=eval_step.name,
    property_file=evaluation_report,
    json_path="classification_metrics.accuracy.value"
)

# check gate
fail_step = FailStep(
    name="FailIfAccuracyLow",
    error_message="Candidate model failed accuracy gate on EMOVO."
)

# register new model
model_metrics = ModelMetrics(
    model_statistics=MetricsSource(
        s3_uri=Join(on="", values=[evaluation_base, "/evaluation.json"]),
        content_type="application/json"
    )
)

register_step = RegisterModel(
    name="RegisterCandidateModel",
    estimator=xgb_estimator,
    model_data=train_step.properties.ModelArtifacts.S3ModelArtifacts,
    content_types=["text/csv", "application/json"],
    response_types=["text/csv", "application/json"],
    inference_instances=["ml.t2.medium", "ml.m5.large", "ml.m5.xlarge"],
    transform_instances=["ml.m5.large", "ml.m5.xlarge"],
    model_package_group_name=p_model_package_group,
    approval_status="Approved",
    model_metrics=model_metrics,
)

cond_step = ConditionStep(
    name="AccuracyGate",
    conditions=[ConditionGreaterThanOrEqualTo(left=accuracy_val, right=p_accuracy_threshold)],
    if_steps=[register_step],
    else_steps=[fail_step],
)

pipeline_name = "ser-emovo-cicd-train-eval-register"

pipeline = Pipeline(
    name=pipeline_name,
    parameters=[
        p_prefix,
        p_processing_instance,
        p_training_instance,
        p_accuracy_threshold,
        p_num_round,
        p_max_depth,
        p_eta,
        p_model_package_group,
    ],
    steps=[preprocess_step, train_step, eval_step, cond_step],
    sagemaker_session=pipeline_sess,
)

print("Built pipeline:", pipeline_name)

INFO:sagemaker.image_uris:Ignoring unnecessary instance type: None.
INFO:sagemaker.image_uris:Defaulting to only available Python version: py3
INFO:sagemaker.image_uris:Defaulting to only available Python version: py3


Built pipeline: ser-emovo-cicd-train-eval-register


In [49]:
# Upsert and run pipeline
pipeline.upsert(role_arn=role)
print("Upserted pipeline:", pipeline_name)

execution = pipeline.start(parameters={
    "Prefix": PREFIX_DEFAULT,
    "ProcessingInstanceType": "ml.t3.large",
    "TrainingInstanceType": "ml.m5.xlarge",
    "AccuracyThreshold": 0.40,
    "NumRound": 100,
    "MaxDepth": 6,
    "Eta": 0.3,
    "ModelPackageGroupName": MODEL_PACKAGE_GROUP
})

print("Execution ARN:", execution.arn)
execution.wait()
print("Execution status:", execution.describe()["PipelineExecutionStatus"])


Upserted pipeline: ser-emovo-cicd-train-eval-register
Execution ARN: arn:aws:sagemaker:us-east-1:472875368112:pipeline/ser-emovo-cicd-train-eval-register/execution/jppklpa9303h
Execution status: Succeeded


In [50]:
# evaluation json
exec_id = execution.arn.split("/")[-1]
eval_s3 = f"s3://{bucket}/{PREFIX_DEFAULT}/evaluation/{exec_id}/evaluation.json"
print("Evaluation JSON:", eval_s3)

Evaluation JSON: s3://sagemaker-us-east-1-472875368112/ser-cicd-emovo/evaluation/jppklpa9303h/evaluation.json


In [51]:
# what was the evaluation result
s3 = boto3.client("s3")

key = f"{PREFIX_DEFAULT}/evaluation/{exec_id}/evaluation.json"

obj = s3.get_object(Bucket=bucket, Key=key)
evaluation = json.loads(obj["Body"].read().decode())

print("\nEvaluation Results:")
print(json.dumps(evaluation, indent=2))

accuracy = evaluation["classification_metrics"]["accuracy"]["value"]
print("\nAccuracy:", accuracy)


Evaluation Results:
{
  "classification_metrics": {
    "accuracy": {
      "value": 0.6078431372549019
    }
  },
  "classification_report": {
    "0": {
      "precision": 0.5555555555555556,
      "recall": 0.625,
      "f1-score": 0.5882352941176471,
      "support": 8
    },
    "1": {
      "precision": 0.38461538461538464,
      "recall": 0.625,
      "f1-score": 0.4761904761904762,
      "support": 8
    },
    "2": {
      "precision": 0.6,
      "recall": 0.6666666666666666,
      "f1-score": 0.631578947368421,
      "support": 9
    },
    "3": {
      "precision": 0.6666666666666666,
      "recall": 0.4444444444444444,
      "f1-score": 0.5333333333333333,
      "support": 9
    },
    "4": {
      "precision": 0.7142857142857143,
      "recall": 0.5555555555555556,
      "f1-score": 0.6250000000000001,
      "support": 9
    },
    "5": {
      "precision": 1.0,
      "recall": 0.75,
      "f1-score": 0.8571428571428571,
      "support": 8
    },
    "accuracy": 0.6078431

In [54]:
if execution.describe()["PipelineExecutionStatus"] != "Succeeded":
    raise RuntimeError("Pipeline failed; not deploying.")

pkg = sm.list_model_packages(
    ModelPackageGroupName=MODEL_PACKAGE_GROUP,
    SortBy="CreationTime",
    SortOrder="Descending",
    MaxResults=1
)["ModelPackageSummaryList"][0]

arn = pkg["ModelPackageArn"]
sm.update_model_package(ModelPackageArn=arn, ModelApprovalStatus="Approved")
print("Approved:", arn)

# create model from model package
mp = sm.describe_model_package(ModelPackageName=arn)
containers = mp["InferenceSpecification"]["Containers"]

model_name = f"ser-model-{int(time.time())}"
sm.create_model(
    ModelName=model_name,
    ExecutionRoleArn=role,
    Containers=[{
        "Image": c["Image"],
        **({"ModelDataUrl": c["ModelDataUrl"]} if "ModelDataUrl" in c else {}),
        **({"Environment": c["Environment"]} if "Environment" in c else {}),
    } for c in containers],
)
print("Created model:", model_name)

# use current instance from existing endpoint
old_cfg_name = sm.describe_endpoint(EndpointName=PROD_ENDPOINT_NAME)["EndpointConfigName"]
old_variant = sm.describe_endpoint_config(EndpointConfigName=old_cfg_name)["ProductionVariants"][0]

cfg_name = f"ser-cfg-{int(time.time())}"
sm.create_endpoint_config(
    EndpointConfigName=cfg_name,
    ProductionVariants=[{
        "VariantName": "AllTraffic",
        "ModelName": model_name,
        "InitialInstanceCount": old_variant["InitialInstanceCount"],
        "InstanceType": old_variant["InstanceType"],
    }]
)

# update endpoint
sm.update_endpoint(EndpointName=PROD_ENDPOINT_NAME, EndpointConfigName=cfg_name)
sm.get_waiter("endpoint_in_service").wait(EndpointName=PROD_ENDPOINT_NAME)

print("Deployed:", model_name, "to", PROD_ENDPOINT_NAME, "using", cfg_name)
print("Rollback config:", old_cfg_name)

Approved: arn:aws:sagemaker:us-east-1:472875368112:model-package/crema-d-emotion-recognition-models-2/4
Created model: ser-model-1771560755
Deployed: ser-model-1771560755 to crema-d-emotion-endpoint-2026-02-20-00-57 using ser-cfg-1771560756
Rollback config: crema-d-emotion-endpoint-2026-02-20-00-57
